In [19]:
import pyreadstat
import pandas as pd

file_path = r"D:\2026\MedCare\ML\Dataset\Raw\randhrs1992_2022v1.sav"

wave_variables = [
    "AGEY_B",
    "HEIGHT",
    "WEIGHT",
    "PMBMI",
    "PMWAIST",
    "BPSYS",
    "BPDIA",
    "BPPULS",
    "SMOKEV",
    "SMOKEN",
    "VGACTX",
    "MDACTX",
    "LTACTX",
    "DRINK",
    "DRINKD",
    "DRINKN",
    "SLEEP",
    "SLEEPE",
    "SLEEPFAL",
    "SLEEPSN",
    "SLEEPBR",
    "DIAB",
    "DIABE",
    "DIABS",
    "HIBP",
    "HIBPS",
    "HIBPE",
    "HEARTE",
    "HEART",
    "HEARTS",
    "STROK",
    "STROKE",
    "STROKS",
    "ARTHR",
    "ARTHRE",
    "ARTHRS",
    "DRUGS",
    "DBORLMED",
    "DBINSULN",
]

selected_columns = []

for w in [13, 14, 15, 16]:

    for variable in wave_variables:
        selected_columns.append(f"R{w}{variable}")

    selected_columns.append(f"H{w}ITOT")

selected_columns.extend([
    "RAGENDER"
])

selected_columns.insert(0, "HHIDPN")

# =========================
# 3. Đọc metadata để kiểm tra cột
# =========================
_, meta = pyreadstat.read_sav(
    file_path,
    metadataonly=True
)

available_columns = set(meta.column_names)

missing_columns = [
    col for col in selected_columns
    if col not in available_columns
]

existing_columns = [
    col for col in selected_columns
    if col in available_columns
]

# =========================
# 4. Thông báo
# =========================
print("Số biến cần:", len(selected_columns))
print("Số biến tìm thấy:", len(existing_columns))
print("Số cột thiếu:", len(missing_columns))

if missing_columns:
    print("Cột thiếu:", missing_columns)
else:
    print("Không có cột thiếu.")

# =========================
# 5. Đọc các cột tồn tại
# =========================
df, meta = pyreadstat.read_sav(
    file_path,
    usecols=existing_columns
)

# =========================
# 6. Tạo cột thiếu, để trống
# =========================
for col in missing_columns:
    df[col] = ""

# Đảm bảo đúng thứ tự cột ban đầu
df = df[selected_columns]

# =========================
# 7. Kiểm tra
# =========================
print("Shape:", df.shape)
print(df.head())

# =========================
# 8. Lưu CSV
# =========================
output_path = r"D:\2026\MedCare\ML\Dataset\Processed\randhrs_selected.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu:", output_path)

Số biến cần: 162
Số biến tìm thấy: 157
Số cột thiếu: 5
Cột thiếu: ['R15PMBMI', 'R15PMWAIST', 'R15BPSYS', 'R15BPDIA', 'R15BPPULS']
Shape: (45234, 162)
       HHIDPN  R13AGEY_B  R13HEIGHT  R13WEIGHT  R13PMBMI  R13PMWAIST  \
0      1010.0        NaN        NaN        NaN       NaN         NaN   
1      2010.0        NaN        NaN        NaN       NaN         NaN   
2      3010.0        NaN        NaN        NaN       NaN         NaN   
3      3020.0        NaN        NaN        NaN       NaN         NaN   
4  10001010.0       76.0     1.8034    63.5026       NaN         NaN   

   R13BPSYS  R13BPDIA  R13BPPULS  R13SMOKEV  ...  R16STROKE  R16STROKS  \
0       NaN       NaN        NaN        NaN  ...        NaN        NaN   
1       NaN       NaN        NaN        NaN  ...        NaN        NaN   
2       NaN       NaN        NaN        NaN  ...        NaN        NaN   
3       NaN       NaN        NaN        NaN  ...        NaN        NaN   
4       NaN       NaN        NaN        0.0  ..

In [20]:
from pathlib import Path
import pandas as pd
import pyreadstat

ROOT = Path(r"D:\2026\MedCare\ML\Dataset")
RAW = ROOT / "Raw"
OUT = ROOT / "Processed"

filter_df = pd.read_csv(ROOT / "Describle\\filter.csv")

rename_map = {
    "HHIDPN": "HHIDPN",
    "EVER HAD HEART ATTACK": "HEARTATTACK_EVER",
    "HEART ATTACK": "HEARTATTACK",
    "HIGH CHOLESTEROL": "HIGHCHOL",
    "RX DRUGS REGULARLY CHOLESTEROL": "CHOLMED",
    "MENOPAUSE FINISHED": "MENOPAUSE",
    "HEART MEDICATION": "HEARTMED",
    "BLOOD PRESSURE MEDICATION": "BPMED",
    "ANGINA": "ANGINA",
    "EVER HAD ANGINA": "ANGINA_EVER",
    "SHORT OF BREATH": "SOB",
    "SHORTNESS OF BREATH": "IS_SOB"
}

wave_prefix = {
    "h22e3a": "R16",
    "h20f1b": "R15",
    "h18f2c": "R14",
    "h16f2c": "R13",
}

waves = {
    "h16f2c": next(c for c in filter_df.columns if c.startswith("h16f2c_metadata")),
    "h18f2c": next(c for c in filter_df.columns if c.startswith("h18f2c_metadata")),
    "h20f1b": next(c for c in filter_df.columns if c.startswith("h20f1b_metadata")),
    "h22e3a": next(c for c in filter_df.columns if c.startswith("h22e3a_metadata")),
}

for wave, meta_col in waves.items():
    cols = (
        filter_df[["Mô tả biến gốc", meta_col]]
        .dropna()
        .assign(code=lambda x: x[meta_col].astype(str).str.replace("*", "", regex=False).str.strip())
        .drop_duplicates("code")
        .set_index("code")["Mô tả biến gốc"]
    )

    sav_path = RAW / f"{wave}.sav"
    _, meta = pyreadstat.read_sav(sav_path, metadataonly=True)
    usecols = [code for code in cols.index if code in meta.column_names]

    df, _ = pyreadstat.read_sav(sav_path, usecols=usecols)

    renamed = {}
    for code, raw_name in cols.items():
        if code not in df.columns:
            continue
        standard = rename_map.get(raw_name, raw_name)
        renamed[code] = "HHIDPN" if standard == "HHIDPN" else f"{wave_prefix[wave]}{standard}"

    df = df.rename(columns=renamed)

    out_path = OUT / f"{wave}_selected.csv"
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"{wave}: {df.shape} -> {out_path}")

h16f2c: (20912, 12) -> D:\2026\MedCare\ML\Dataset\Processed\h16f2c_selected.csv
h18f2c: (17146, 12) -> D:\2026\MedCare\ML\Dataset\Processed\h18f2c_selected.csv
h20f1b: (15723, 12) -> D:\2026\MedCare\ML\Dataset\Processed\h20f1b_selected.csv
h22e3a: (15856, 12) -> D:\2026\MedCare\ML\Dataset\Processed\h22e3a_selected.csv


In [21]:
import pandas as pd
from pathlib import Path

folder = Path(r"d:\2026\MedCare\ML\Dataset\Processed")

files = [
    folder / "randhrs_selected.csv",
    folder / "h22e3a_selected.csv",
    folder / "h20f1b_selected.csv",
    folder / "h18f2c_selected.csv",
    folder / "h16f2c_selected.csv",
]

files = [f for f in files if f.exists()]

dfs = []
for path in files:
    df = pd.read_csv(path)
    if "HHIDPN" not in df.columns:
        raise ValueError(f"{path.name} không có HHIDPN")
    df["HHIDPN"] = pd.to_numeric(df["HHIDPN"], errors="coerce")
    df = df.dropna(subset=["HHIDPN"]).drop_duplicates(subset=["HHIDPN"])
    dfs.append(df)

common_ids = set(dfs[0]["HHIDPN"])
for df in dfs[1:]:
    common_ids &= set(df["HHIDPN"])

filtered = [df[df["HHIDPN"].isin(common_ids)].copy() for df in dfs]

merged = filtered[0]
for i, df in enumerate(filtered[1:], start=1):
    merged = merged.merge(df, on="HHIDPN", how="inner", suffixes=("", f"_{i}"))

print("Số mẫu chung:", len(merged))
print("Shape:", merged.shape)
print(merged.head())

out_path = OUT / "merged.csv"
merged.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Đã lưu:", out_path)

Số mẫu chung: 11158
Shape: (11158, 206)
       HHIDPN  R13AGEY_B  R13HEIGHT  R13WEIGHT  R13PMBMI  R13PMWAIST  \
0  10003030.0       60.0     1.5494   86.18210       NaN         NaN   
1  10004040.0       70.0     1.6510   68.03850       NaN         NaN   
2  10013040.0       68.0     1.5748   59.42029       NaN         NaN   
3  10038010.0       79.0     1.7526   72.57440       NaN         NaN   
4  10038040.0       73.0     1.6764   64.40978       NaN         NaN   

   R13BPSYS  R13BPDIA  R13BPPULS  R13SMOKEV  ...  R13HEARTMED  \
0       NaN       NaN        NaN        0.0  ...          NaN   
1       NaN       NaN        NaN        1.0  ...          NaN   
2       NaN       NaN        NaN        1.0  ...          NaN   
3       NaN       NaN        NaN        0.0  ...          1.0   
4       NaN       NaN        NaN        1.0  ...          NaN   

   R13HEARTATTACK_EVER  R13HEARTATTACK  R13ANGINA_EVER  R13ANGINA  \
0                  NaN             NaN             NaN        NaN  

In [22]:
import pandas as pd
from pathlib import Path
import re

path = Path(r"D:\2026\MedCare\ML\Dataset\Processed\merged.csv")
df = pd.read_csv(path)

cols = [c for c in df.columns if c != "HHIDPN"]

rows = []
for p in ["R", "H"]:
    for x in [13, 14, 15, 16]:
        prefix = f"{p}{x}"
        n = sum(1 for c in cols if re.match(rf"^{re.escape(prefix)}", c))
        rows.append({"wave": prefix, "n_variables": n})

out = pd.DataFrame(rows)
print(out)

  wave  n_variables
0  R13           50
1  R14           50
2  R15           50
3  R16           50
4  H13            1
5  H14            1
6  H15            1
7  H16            1


In [23]:
import pandas as pd
import re

df = pd.read_csv(r"D:\2026\MedCare\ML\Dataset\Processed\merged.csv")
cols = [c for c in df.columns]

for prefix in ["R15", "H15"]:
    wave_cols = [c for c in cols if re.match(rf"^{re.escape(prefix)}", c)]
    print(f"{prefix}: {len(wave_cols)} biến")
    print(wave_cols)
    print("-" * 80)

R15: 50 biến
['R15AGEY_B', 'R15HEIGHT', 'R15WEIGHT', 'R15PMBMI', 'R15PMWAIST', 'R15BPSYS', 'R15BPDIA', 'R15BPPULS', 'R15SMOKEV', 'R15SMOKEN', 'R15VGACTX', 'R15MDACTX', 'R15LTACTX', 'R15DRINK', 'R15DRINKD', 'R15DRINKN', 'R15SLEEP', 'R15SLEEPE', 'R15SLEEPFAL', 'R15SLEEPSN', 'R15SLEEPBR', 'R15DIAB', 'R15DIABE', 'R15DIABS', 'R15HIBP', 'R15HIBPS', 'R15HIBPE', 'R15HEARTE', 'R15HEART', 'R15HEARTS', 'R15STROK', 'R15STROKE', 'R15STROKS', 'R15ARTHR', 'R15ARTHRE', 'R15ARTHRS', 'R15DRUGS', 'R15DBORLMED', 'R15DBINSULN', 'R15BPMED', 'R15HEARTMED', 'R15HEARTATTACK_EVER', 'R15HEARTATTACK', 'R15ANGINA_EVER', 'R15ANGINA', 'R15HIGHCHOL', 'R15IS_SOB', 'R15SOB', 'R15CHOLMED', 'R15MENOPAUSE']
--------------------------------------------------------------------------------
H15: 1 biến
['H15ITOT']
--------------------------------------------------------------------------------


In [24]:
import pandas as pd

df = pd.read_csv(r"D:\2026\MedCare\ML\Dataset\Processed\merged.csv")
missing = df.isna().sum().sort_values(ascending=False)

print("Shape:", df.shape)
print("\nMissing value mỗi cột:")
print(missing.to_string())

Shape: (11158, 206)

Missing value mỗi cột:
R16ANGINA_EVER         11158
R15BPPULS              11158
R15BPSYS               11158
R15BPDIA               11158
R15PMWAIST             11158
R15PMBMI               11158
R16HEARTATTACK_EVER    11158
R16IS_SOB              11154
R14IS_SOB              11148
R15HEARTATTACK_EVER    11148
R15ANGINA_EVER         11147
R13HEARTATTACK_EVER    10893
R13ANGINA_EVER         10893
R13MENOPAUSE           10834
R14ANGINA              10355
R14HEARTATTACK         10179
R15MENOPAUSE            9755
R16ANGINA               9631
R16HEARTATTACK          9631
R14HEARTATTACK_EVER     9625
R14ANGINA_EVER          9625
R13HEARTATTACK          9528
R13ANGINA               9493
R14DBINSULN             9056
R13HEARTMED             9011
R15ANGINA               8968
R15HEARTATTACK          8968
R13DBINSULN             8894
R14HEARTMED             8710
R15DBINSULN             8659
R15HEARTMED             8483
R13DBORLMED             8438
R16DBINSULN             8285

In [25]:
import pandas as pd
import re

df = pd.read_csv(r"D:\2026\MedCare\ML\Dataset\Processed\merged.csv")

cols = [c for c in df.columns]

for c in cols:
    print(f"\n=== {c} ===") 
    print("Unique values:", sorted(pd.Series(df[c].dropna().unique()).astype(str).tolist()))
    


=== HHIDPN ===
Unique values: ['10003030.0', '10004040.0', '10013040.0', '10038010.0', '10038040.0', '10059020.0', '10059030.0', '10075020.0', '10106020.0', '10299010.0', '10325020.0', '10372010.0', '10404010.0', '10451010.0', '10451020.0', '10458020.0', '10458030.0', '10477010.0', '10477020.0', '10481020.0', '10537011.0', '10571010.0', '10577010.0', '10648010.0', '10648020.0', '10740010.0', '10769040.0', '10773020.0', '10773030.0', '10818010.0', '10818040.0', '10893010.0', '10962010.0', '10989010.0', '10989020.0', '110096010.0', '110163010.0', '110398020.0', '110427010.0', '110427020.0', '110455010.0', '110455020.0', '11067020.0', '11067030.0', '11071020.0', '110780020.0', '110942010.0', '111363010.0', '111363020.0', '11141010.0', '11141020.0', '111502010.0', '111502020.0', '111597010.0', '111597020.0', '111640010.0', '111640020.0', '111666020.0', '111666021.0', '111939010.0', '111997020.0', '112014020.0', '11230010.0', '112432010.0', '112432011.0', '112432020.0', '112450010.0', '112

In [26]:
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

INPUT = r"D:\2026\MedCare\ML\Dataset\Processed\merged.csv"
OUTPUT = r"D:\2026\MedCare\ML\Dataset\Processed\merged_imputed.csv"

WAVES = [13, 14, 15, 16]

def wc(w, s):
    return f"R{w}{s}"


# ============================================================
# 1. LOAD
# ============================================================

df = pd.read_csv(INPUT)


# ============================================================
# 2. CLEAN MISSING CODES
# ============================================================

# Chỉ các biến mà 8/9 là DK/RF
dk_rf = [
    "VGACTX", "MDACTX", "LTACTX",
    "SLEEP", "SLEEPFAL", "SLEEPSN", "SLEEPBR",
    "HIBP", "HIBPS", "DIAB", "DIABS",
    "HEART", "HEARTS", "STROK", "STROKS",
    "ARTHR", "ARTHRS", "SOB", "IS_SOB",
    "SMOKEN", "DRINK", "SLEEPE", "HIGHCHOL",
    "MENOPAUSE", "DBORLMED", "DBINSULN",
    "CHOLMED", "HEARTMED", "BPMED", "DRUGS"
]

for w in WAVES:
    for s in dk_rf:
        c = wc(w, s)
        if c in df:
            df[c] = df[c].replace(
                [-1, -2, -3, -4, -5, -6, -7, -8, -9, 8, 9],
                np.nan
            )


# ============================================================
# 3. HEIGHT: LOCF + NOCB
# ============================================================

cols = [wc(w, "HEIGHT") for w in WAVES if wc(w, "HEIGHT") in df]

if cols:
    df[cols] = df[cols].ffill(axis=1).bfill(axis=1)


# ============================================================
# 4. WEIGHT: LINEAR INTERPOLATION
# ============================================================

cols = [wc(w, "WEIGHT") for w in WAVES if wc(w, "WEIGHT") in df]

if cols:
    df[cols] = df[cols].interpolate(
        axis=1,
        method="linear"
        # limit_area="inside"
    )


# ============================================================
# 5. EVER: ONCE 1 -> ALWAYS 1
# ============================================================

ever_vars = [
    "DIABE", "HIBPE", "HEARTE", "STROKE", "ARTHRE",
    "SLEEPE", "HEARTATTACK_EVER", "ANGINA_EVER", "SMOKEV"
]

for s in ever_vars:
    cols = [wc(w, s) for w in WAVES if wc(w, s) in df]

    if not cols:
        continue

    has_ever = df[cols].eq(1).any(axis=1)

    for c in cols:
        df.loc[has_ever & df[c].isna(), c] = 1


# ============================================================
# 6. EVER = 0 -> CURRENT = 0
# ============================================================

ever_current = {
    "DIABE": ["DIAB", "DIABS"],
    "HIBPE": ["HIBP", "HIBPS"],
    "HEARTE": ["HEART", "HEARTS"],
    "ARTHRE": ["ARTHR", "ARTHRS"]
}

for ever, currents in ever_current.items():
    for w in WAVES:
        e = wc(w, ever)

        if e not in df:
            continue

        for s in currents:
            c = wc(w, s)

            if c in df:
                mask = df[e].eq(0) & df[c].isna()
                df.loc[mask, c] = 0


# ============================================================
# 7. MEDICATION = 1 -> DISEASE = 1
# ============================================================

med_disease = {
    "DBORLMED": ["DIAB", "DIABS"],
    "DBINSULN": ["DIAB", "DIABS"],
    "BPMED": ["HIBP", "HIBPS"],
    "HEARTMED": ["HEART", "HEARTS"]
}

for med, diseases in med_disease.items():
    for w in WAVES:
        m = wc(w, med)

        if m not in df:
            continue

        for disease in diseases:
            c = wc(w, disease)

            if c in df:
                mask = df[m].eq(1) & df[c].isna()
                df.loc[mask, c] = 1


# ============================================================
# 8. SMOKING
# ============================================================

for w in WAVES:
    ever = wc(w, "SMOKEV")
    current = wc(w, "SMOKEN")

    if ever in df and current in df:
        mask = df[ever].eq(0) & df[current].isna()
        df.loc[mask, current] = 0


# ============================================================
# 9. ALCOHOL
# ============================================================

for w in WAVES:
    drink = wc(w, "DRINK")
    days = wc(w, "DRINKD")
    amount = wc(w, "DRINKN")

    if drink not in df:
        continue

    if days in df:
        mask = df[drink].eq(0) & df[days].isna()
        df.loc[mask, days] = 0

    if amount in df:
        mask = df[drink].eq(0) & df[amount].isna()
        df.loc[mask, amount] = 0


# ============================================================
# 10. MENOPAUSE
# ============================================================

if "RAGENDER" in df:
    for w in WAVES:
        c = wc(w, "MENOPAUSE")

        if c in df:
            mask = df["RAGENDER"].eq(1) & df[c].isna()
            df.loc[mask, c] = 0


# ============================================================
# 11. VITAL SIGNS: INVALID -> NaN
# ============================================================

limits = {
    "BPSYS": (40, 300),
    "BPDIA": (20, 200),
    "BPPULS": (20, 250),
    "PMWAIST": (30, 250)
}

for s, (low, high) in limits.items():
    for w in WAVES:
        c = wc(w, s)

        if c in df:
            mask = (
                df[c].notna()
                & ((df[c] < low) | (df[c] >= 999) | (df[c] > high))
            )
            df.loc[mask, c] = np.nan


# ============================================================
# 12. BMI = WEIGHT / HEIGHT²
# ============================================================

for w in WAVES:
    h = wc(w, "HEIGHT")
    wt = wc(w, "WEIGHT")
    bmi = wc(w, "PMBMI")

    if h in df and wt in df:
        if bmi not in df:
            df[bmi] = np.nan

        mask = (
            df[h].notna()
            & df[wt].notna()
            & (df[h] > 0)
            & (df[wt] > 0)
            & df[bmi].isna()
        )

        calculated = df[wt] / (df[h] ** 2)
        df.loc[mask, bmi] = calculated[mask]


# ============================================================
# 13. DROP: THIẾU TOÀN BỘ HEIGHT + WEIGHT
# ============================================================

anthro = [
    wc(w, s)
    for w in WAVES
    for s in ["HEIGHT", "WEIGHT"]
    if wc(w, s) in df
]

if anthro:
    df = df.loc[~df[anthro].isna().all(axis=1)].copy()


# ============================================================
# 14. SAVE
# ============================================================

df.to_csv(OUTPUT, index=False)

print(f"Done: {OUTPUT}")
print(f"Shape: {df.shape}")

Done: D:\2026\MedCare\ML\Dataset\Processed\merged_imputed.csv
Shape: (11158, 206)


In [27]:
import pandas as pd

path_raw = r"D:\2026\MedCare\ML\Dataset\Processed\merged.csv"
path_clean = r"D:\2026\MedCare\ML\Dataset\Processed\merged_imputed.csv"

raw = pd.read_csv(path_raw)
clean = pd.read_csv(path_clean)

cmp = pd.concat(
    [raw.isna().sum(), clean.isna().sum()],
    axis=1,
    keys=["missing_merged", "missing_merged_imputed"]
).fillna(0).astype(int)

cmp["missing_them_o_merged"] = cmp["missing_merged"] - cmp["missing_merged_imputed"]
cmp = cmp.sort_values("missing_merged", ascending=False)

print("Shape merged.csv:", raw.shape)
print("Shape merged_imputed.csv:", clean.shape)
print("\nMissing value mỗi cột (so sánh cạnh nhau):")
print(cmp.to_string())

print("\nTổng missing:")
print(cmp.sum().to_string())

# Lưu bảng so sánh theo cột
out_cmp = r"D:\2026\MedCare\ML\Dataset\Processed\missing_compare.csv"
cmp.to_csv(out_cmp, encoding="utf-8-sig", index=True, index_label="column")

# Lưu tổng missing
out_total = r"D:\2026\MedCare\ML\Dataset\Processed\missing_compare_total.csv"
cmp.sum().to_frame("total").to_csv(out_total, encoding="utf-8-sig", index=True, index_label="metric")

print("Đã lưu:", out_cmp)
print("Đã lưu:", out_total)

Shape merged.csv: (11158, 206)
Shape merged_imputed.csv: (11158, 206)

Missing value mỗi cột (so sánh cạnh nhau):
                     missing_merged  missing_merged_imputed  missing_them_o_merged
R16ANGINA_EVER                11158                   10923                    235
R15BPPULS                     11158                   11158                      0
R15BPSYS                      11158                   11158                      0
R15BPDIA                      11158                   11158                      0
R15PMWAIST                    11158                   11158                      0
R15PMBMI                      11158                      41                  11117
R16HEARTATTACK_EVER           11158                   10666                    492
R16IS_SOB                     11154                   11154                      0
R14IS_SOB                     11148                   11148                      0
R15HEARTATTACK_EVER           11148                   10

In [28]:
import os
import pandas as pd


def create_labels(df):
    t2d = ["DIAB", "DIABS", "DIABE", "DBORLMED", "DBINSULN"]
    cvd = [
        "HEART", "HEARTS", "HEARTE",
        "HEARTATTACK", "HEARTATTACK_EVER",
        "ANGINA", "ANGINA_EVER", "HEARTMED",
        "STROK", "STROKS", "STROKE"
    ]

    for w in range(13, 17):
        for name, variables in [("T2D", t2d), ("CVD", cvd)]:
            cols = [f"R{w}{v}" for v in variables if f"R{w}{v}" in df.columns]
            values = df[cols]

            df[f"R{w}{name}"] = (
                values.eq(1).any(axis=1).astype("int8")
            )

            # Tất cả biến thiếu → không xác định
            df.loc[values.isna().all(axis=1), f"R{w}{name}"] = pd.NA

    # Xóa biến dùng để tạo nhãn
    used_cols = [
        f"R{w}{v}"
        for w in range(13, 17)
        for v in t2d + cvd
    ]
    df.drop(columns=used_cols, errors="ignore", inplace=True)

    return df


# Paths
input_path = r"D:\2026\MedCare\ML\Dataset\Processed\merged_imputed.csv"
output_path = r"D:\2026\MedCare\ML\Dataset\Clean\merged_labeled.csv"

# Process
df = pd.read_csv(input_path)
df = create_labels(df)
df.to_csv(output_path, index=False)

print(f"Đã lưu: {output_path}")

C:\Users\dattr\AppData\Local\Temp\ipykernel_2848\1136907300.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"R{w}{name}"] = (
C:\Users\dattr\AppData\Local\Temp\ipykernel_2848\1136907300.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"R{w}{name}"] = (
C:\Users\dattr\AppData\Local\Temp\ipykernel_2848\1136907300.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.conc

Đã lưu: D:\2026\MedCare\ML\Dataset\Clean\merged_labeled.csv


# Split Wave

In [29]:
import pandas as pd
from pathlib import Path

INPUT = Path(r"D:\2026\MedCare\ML\Dataset\Clean\merged_labeled.csv")
OUTPUT = INPUT.with_name("ml_dataset_long.csv")

df = pd.read_csv(INPUT)

waves = [13, 14, 15, 16]
labels = ["T2D", "CVD"]

# Tự động lấy feature: các biến R{wave}...,
# nhưng loại label và HHIDPN/RAGENDER
wave_cols = {
    w: [
        c for c in df.columns
        if c.startswith(f"R{w}")
        and c not in [f"R{w}T2D", f"R{w}CVD"]
    ]
    for w in waves
}

def make_wave(df, w):
    cols = ["HHIDPN", "RAGENDER"] + wave_cols[w]

    # Thêm biến hộ gia đình
    hcol = f"H{w}ITOT"
    if hcol in df.columns:
        cols.append(hcol)

    # Thêm label
    cols += [f"R{w}T2D", f"R{w}CVD"]

    data = df[[c for c in cols if c in df.columns]].copy()

    # Bỏ tiền tố wave
    data.columns = [
        c.replace(f"R{w}", "", 1).replace(f"H{w}", "", 1)
        for c in data.columns
    ]

    data["WAVE"] = w
    return data

result = pd.concat(
    [make_wave(df, w) for w in waves],
    ignore_index=True
)

result.to_csv(OUTPUT, index=False)

print(f"Đã lưu: {OUTPUT}")
print(f"Kích thước: {result.shape}")

Đã lưu: D:\2026\MedCare\ML\Dataset\Clean\ml_dataset_long.csv
Kích thước: (44632, 40)


In [30]:
import pandas as pd

path = r"D:\2026\MedCare\ML\Dataset\Clean\ml_dataset_long.csv"

df = pd.read_csv(path)

for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")

1. HHIDPN
2. RAGENDER
3. AGEY_B
4. HEIGHT
5. WEIGHT
6. PMBMI
7. PMWAIST
8. BPSYS
9. BPDIA
10. BPPULS
11. SMOKEV
12. SMOKEN
13. VGACTX
14. MDACTX
15. LTACTX
16. DRINK
17. DRINKD
18. DRINKN
19. SLEEP
20. SLEEPE
21. SLEEPFAL
22. SLEEPSN
23. SLEEPBR
24. HIBP
25. HIBPS
26. HIBPE
27. ARTHR
28. ARTHRE
29. ARTHRS
30. DRUGS
31. BPMED
32. HIGHCHOL
33. IS_SOB
34. SOB
35. CHOLMED
36. MENOPAUSE
37. ITOT
38. T2D
39. CVD
40. WAVE


In [31]:
import pandas as pd
from pathlib import Path

INPUT = Path(r"D:\2026\MedCare\ML\Dataset\Clean\ml_dataset_long.csv")
OUT = INPUT.parent

T2D_FEATURES = [
    "RAGENDER", "AGEY_B", "HEIGHT", "WEIGHT", "PMBMI", "PMWAIST",
    "SMOKEV", "SMOKEN", "VGACTX", "MDACTX", "LTACTX",
    "DRINK", "DRINKD", "DRINKN",
    "SLEEP", "SLEEPE", "SLEEPFAL", "SLEEPSN", "SLEEPBR",
    "HIBP", "HIBPS", "HIBPE", "ARTHR", "ARTHRE", "ARTHRS",
    "DRUGS", "BPMED", "HIGHCHOL", "CHOLMED", "MENOPAUSE", "ITOT"
]

CVD_FEATURES = [
    "RAGENDER", "AGEY_B", "HEIGHT", "WEIGHT", "PMBMI", "PMWAIST",
    "BPSYS", "BPDIA", "BPPULS",
    "SMOKEV", "SMOKEN", "VGACTX", "MDACTX", "LTACTX",
    "DRINK", "DRINKD", "DRINKN",
    "SLEEP", "SLEEPE", "SLEEPFAL", "SLEEPSN", "SLEEPBR",
    "HIBP", "HIBPS", "HIBPE", "ARTHR", "ARTHRE", "ARTHRS",
    "DRUGS", "BPMED", "HIGHCHOL", "IS_SOB", "SOB",
    "CHOLMED", "MENOPAUSE", "ITOT"
]

df = pd.read_csv(INPUT)

t2d = df[["HHIDPN"] + T2D_FEATURES + ["T2D", "WAVE"]]
cvd = df[["HHIDPN"] + CVD_FEATURES + ["CVD", "WAVE"]]

t2d.to_csv(OUT / "t2d_dataset.csv", index=False)
cvd.to_csv(OUT / "cvd_dataset.csv", index=False)

print("T2D:", t2d.shape)
print("CVD:", cvd.shape)
print("Đã lưu:", OUT)

T2D: (44632, 34)
CVD: (44632, 39)
Đã lưu: D:\2026\MedCare\ML\Dataset\Clean


In [32]:
from sklearn.model_selection import GroupShuffleSplit

DATA_DIR = Path(r"D:\2026\MedCare\ML\Dataset\Clean")
TRAIN_DIR = DATA_DIR / "Train"
TEST_DIR = DATA_DIR / "Test"

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

def split_by_person(path, target):
    df = pd.read_csv(path)

    X = df.drop(columns=[target])
    y = df[target]
    groups = df["HHIDPN"]

    split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(split.split(X, y, groups))

    train = df.iloc[train_idx].copy()
    test = df.iloc[test_idx].copy()

    return train, test


t2d_train, t2d_test = split_by_person(
    DATA_DIR / "t2d_dataset.csv",
    "T2D"
)

cvd_train, cvd_test = split_by_person(
    DATA_DIR / "cvd_dataset.csv",
    "CVD"
)


t2d_train.to_csv(
    TRAIN_DIR / "t2d_train.csv",
    index=False
)

t2d_test.to_csv(
    TEST_DIR / "t2d_test.csv",
    index=False
)

cvd_train.to_csv(
    TRAIN_DIR / "cvd_train.csv",
    index=False
)

cvd_test.to_csv(
    TEST_DIR / "cvd_test.csv",
    index=False
)

print("T2D:", t2d_train.shape, t2d_test.shape)
print("CVD:", cvd_train.shape, cvd_test.shape)

T2D: (35704, 34) (8928, 34)
CVD: (35704, 39) (8928, 39)


In [33]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r"D:\2026\MedCare\ML\Dataset\Clean\Train")

T2D_FEATURES = [
    "RAGENDER", "AGEY_B", "HEIGHT", "WEIGHT", "PMBMI", "PMWAIST",
    "SMOKEV", "SMOKEN", "VGACTX", "MDACTX", "LTACTX",
    "DRINK", "DRINKD", "DRINKN",
    "SLEEP", "SLEEPE", "SLEEPFAL", "SLEEPSN", "SLEEPBR",
    "HIBP", "HIBPS", "HIBPE", "ARTHR", "ARTHRE", "ARTHRS",
    "DRUGS", "BPMED", "HIGHCHOL", "CHOLMED", "MENOPAUSE", "ITOT"
]

CVD_FEATURES = [
    "RAGENDER", "AGEY_B", "HEIGHT", "WEIGHT", "PMBMI", "PMWAIST",
    "BPSYS", "BPDIA", "BPPULS",
    "SMOKEV", "SMOKEN", "VGACTX", "MDACTX", "LTACTX",
    "DRINK", "DRINKD", "DRINKN",
    "SLEEP", "SLEEPE", "SLEEPFAL", "SLEEPSN", "SLEEPBR",
    "HIBP", "HIBPS", "HIBPE", "ARTHR", "ARTHRE", "ARTHRS",
    "DRUGS", "BPMED", "HIGHCHOL", "IS_SOB", "SOB",
    "CHOLMED", "MENOPAUSE", "ITOT"
]

COMMON_FE = [
    "FE_WtHR",
    "FE_BMI_Waist",
    "FE_Weekly_Drinks",
    "FE_Smoking_Status",
    "FE_Physical_Activity_Score",
    "FE_Sleep_Disturbance_Index",
    "FE_Log_Income",
    "FE_Menopause_Age"
]

CVD_FE = [
    "FE_Pulse_Pressure",
    "FE_MAP",
    "FE_RPP"
]


def engineer_features(df, disease):
    df = df.copy()

    # Thể trạng
    df["FE_WtHR"] = df["PMWAIST"] / (df["HEIGHT"] * 100)
    df["FE_BMI_Waist"] = df["PMBMI"] * df["PMWAIST"]

    # Rượu bia
    df["FE_Weekly_Drinks"] = (
        df["DRINKD"] * df["DRINKN"]
    ).where(df["DRINK"].eq(1), 0)

    # Hút thuốc
    df["FE_Smoking_Status"] = np.select(
        [
            df["SMOKEV"].eq(0),
            df["SMOKEV"].eq(1) & df["SMOKEN"].eq(0),
            df["SMOKEV"].eq(1) & df["SMOKEN"].eq(1)
        ],
        [0, 1, 2],
        default=np.nan
    )

    # Vận động
    df["FE_Physical_Activity_Score"] = (
        (6 - df["VGACTX"]) * 3
        + (6 - df["MDACTX"]) * 2
        + (6 - df["LTACTX"])
    )

    # Giấc ngủ
    df["FE_Sleep_Disturbance_Index"] = (
        df["SLEEP"]
        + df["SLEEPFAL"]
        + df["SLEEPSN"]
        + df["SLEEPBR"]
    )

    # Thu nhập
    df["FE_Log_Income"] = np.log1p(df["ITOT"].clip(lower=0))

    # Đặc trưng riêng CVD
    if disease == "CVD":
        df["FE_Pulse_Pressure"] = df["BPSYS"] - df["BPDIA"]
        df["FE_MAP"] = df["BPDIA"] + df["FE_Pulse_Pressure"] / 3
        df["FE_RPP"] = df["BPSYS"] * df["BPPULS"]

    return df


def process_file(filename, disease, features, target):
    df = pd.read_csv(BASE / filename)
    df = engineer_features(df, disease)

    fe_cols = COMMON_FE + (CVD_FE if disease == "CVD" else [])
    cols = features + fe_cols + [target, "HHIDPN", "WAVE"]

    df = df[[c for c in cols if c in df.columns]]
    output = BASE / f"{disease.lower()}_train_engineered.csv"
    df.to_csv(output, index=False)

    print(f"{disease}: {df.shape}")
    print(f"Saved: {output}")


process_file(
    "t2d_train.csv",
    "T2D",
    T2D_FEATURES,
    "T2D"
)

process_file(
    "cvd_train.csv",
    "CVD",
    CVD_FEATURES,
    "CVD"
)

T2D: (35704, 41)
Saved: D:\2026\MedCare\ML\Dataset\Clean\Train\t2d_train_engineered.csv
CVD: (35704, 49)
Saved: D:\2026\MedCare\ML\Dataset\Clean\Train\cvd_train_engineered.csv


In [34]:
BASE = Path(r"D:\2026\MedCare\ML\Dataset\Clean\Test")
process_file(
    "t2d_test.csv",
    "T2D",
    T2D_FEATURES,
    "T2D"
)

process_file(
    "cvd_test.csv",
    "CVD",
    CVD_FEATURES,
    "CVD"
)

T2D: (8928, 41)
Saved: D:\2026\MedCare\ML\Dataset\Clean\Test\t2d_train_engineered.csv
CVD: (8928, 49)
Saved: D:\2026\MedCare\ML\Dataset\Clean\Test\cvd_train_engineered.csv


# Future

In [35]:
import pandas as pd
from pathlib import Path

INPUT = Path(r"D:\2026\MedCare\ML\Dataset\Clean\merged_labeled.csv")
OUT = INPUT.parent

waves = [13, 14, 15, 16]

T2D_FEATURES = [
    "AGEY_B", "HEIGHT", "WEIGHT", "PMBMI", "PMWAIST",
    "SMOKEV", "SMOKEN", "VGACTX", "MDACTX", "LTACTX",
    "DRINK", "DRINKD", "DRINKN",
    "SLEEP", "SLEEPE", "SLEEPFAL", "SLEEPSN", "SLEEPBR",
    "HIBP", "HIBPS", "HIBPE", "ARTHR", "ARTHRE", "ARTHRS",
    "DRUGS", "BPMED", "HIGHCHOL", "CHOLMED", "MENOPAUSE"
]

CVD_FEATURES = [
    "AGEY_B", "HEIGHT", "WEIGHT", "PMBMI", "PMWAIST",
    "BPSYS", "BPDIA", "BPPULS",
    "SMOKEV", "SMOKEN", "VGACTX", "MDACTX", "LTACTX",
    "DRINK", "DRINKD", "DRINKN",
    "SLEEP", "SLEEPE", "SLEEPFAL", "SLEEPSN", "SLEEPBR",
    "HIBP", "HIBPS", "HIBPE", "ARTHR", "ARTHRE", "ARTHRS",
    "DRUGS", "BPMED", "HIGHCHOL", "IS_SOB", "SOB",
    "CHOLMED", "MENOPAUSE"
]

def make_wave_features(base_cols, wave):
    return [f"R{wave}{c}" for c in base_cols] + [f"H{wave}ITOT"]

df = pd.read_csv(INPUT)

# T2D
t2d_cols = ["HHIDPN", "RAGENDER"]
t2d_cols += [col for wave in waves for col in make_wave_features(T2D_FEATURES, wave)]
t2d_cols += [f"R{wave}T2D" for wave in waves]

# CVD
cvd_cols = ["HHIDPN", "RAGENDER"]
cvd_cols += [col for wave in waves for col in make_wave_features(CVD_FEATURES, wave)]
cvd_cols += [f"R{wave}CVD" for wave in waves]

t2d = df[t2d_cols].copy()
cvd = df[cvd_cols].copy()

t2d.to_csv(OUT / "t2d_dataset_ft.csv", index=False)
cvd.to_csv(OUT / "cvd_dataset_ft.csv", index=False)

print("T2D:", t2d.shape)
print("CVD:", cvd.shape)
print("Đã lưu tại:", OUT)

T2D: (11158, 126)
CVD: (11158, 146)
Đã lưu tại: D:\2026\MedCare\ML\Dataset\Clean


In [36]:
import pandas as pd

def keep_wave16_target_only(df, disease):
    keep = ["HHIDPN", "RAGENDER"]
    removed_cols = []

    for col in df.columns:
        if col in keep:
            continue

        if col == f"R16{disease}":
            keep.append(col)
            continue

        # Giữ feature từ wave 13-15 và ITOT có tiền tố H
        if col.startswith(("R13", "R14", "R15")) or col.startswith(("H13", "H14", "H15")):
            keep.append(col)
            continue

        removed_cols.append(col)

    cleaned = df[keep].copy()
    return cleaned, removed_cols


# T2D
t2d = pd.read_csv(r"D:\2026\MedCare\ML\Dataset\Clean\t2d_dataset_ft.csv")
t2d, t2d_removed = keep_wave16_target_only(t2d, "T2D")
t2d.to_csv(r"D:\2026\MedCare\ML\Dataset\Clean\t2d_dataset_ft.csv", index=False)

print("T2D removed columns:", len(t2d_removed))
print(t2d_removed)

# CVD
cvd = pd.read_csv(r"D:\2026\MedCare\ML\Dataset\Clean\cvd_dataset_ft.csv")
cvd, cvd_removed = keep_wave16_target_only(cvd, "CVD")
cvd.to_csv(r"D:\2026\MedCare\ML\Dataset\Clean\cvd_dataset_ft.csv", index=False)

print("CVD removed columns:", len(cvd_removed))
print(cvd_removed)

print("T2D:", t2d.shape)
print("CVD:", cvd.shape)

T2D removed columns: 30
['R16AGEY_B', 'R16HEIGHT', 'R16WEIGHT', 'R16PMBMI', 'R16PMWAIST', 'R16SMOKEV', 'R16SMOKEN', 'R16VGACTX', 'R16MDACTX', 'R16LTACTX', 'R16DRINK', 'R16DRINKD', 'R16DRINKN', 'R16SLEEP', 'R16SLEEPE', 'R16SLEEPFAL', 'R16SLEEPSN', 'R16SLEEPBR', 'R16HIBP', 'R16HIBPS', 'R16HIBPE', 'R16ARTHR', 'R16ARTHRE', 'R16ARTHRS', 'R16DRUGS', 'R16BPMED', 'R16HIGHCHOL', 'R16CHOLMED', 'R16MENOPAUSE', 'H16ITOT']
CVD removed columns: 35
['R16AGEY_B', 'R16HEIGHT', 'R16WEIGHT', 'R16PMBMI', 'R16PMWAIST', 'R16BPSYS', 'R16BPDIA', 'R16BPPULS', 'R16SMOKEV', 'R16SMOKEN', 'R16VGACTX', 'R16MDACTX', 'R16LTACTX', 'R16DRINK', 'R16DRINKD', 'R16DRINKN', 'R16SLEEP', 'R16SLEEPE', 'R16SLEEPFAL', 'R16SLEEPSN', 'R16SLEEPBR', 'R16HIBP', 'R16HIBPS', 'R16HIBPE', 'R16ARTHR', 'R16ARTHRE', 'R16ARTHRS', 'R16DRUGS', 'R16BPMED', 'R16HIGHCHOL', 'R16IS_SOB', 'R16SOB', 'R16CHOLMED', 'R16MENOPAUSE', 'H16ITOT']
T2D: (11158, 96)
CVD: (11158, 111)


In [37]:
from sklearn.model_selection import GroupShuffleSplit

DATA_DIR = Path(r"D:\2026\MedCare\ML\Dataset\Clean")
TRAIN_DIR = DATA_DIR / "Train"
TEST_DIR = DATA_DIR / "Test"

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

def split_by_person(path, target):
    df = pd.read_csv(path)

    X = df.drop(columns=[target])
    y = df[target]
    groups = df["HHIDPN"]

    split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(split.split(X, y, groups))

    train = df.iloc[train_idx].copy()
    test = df.iloc[test_idx].copy()

    return train, test


t2d_train, t2d_test = split_by_person(
    DATA_DIR / "t2d_dataset_ft.csv",
    "R16T2D"
)

cvd_train, cvd_test = split_by_person(
    DATA_DIR / "cvd_dataset_ft.csv",
    "R16CVD"
)


t2d_train.to_csv(
    TRAIN_DIR / "t2d_train_ft.csv",
    index=False
)

t2d_test.to_csv(
    TEST_DIR / "t2d_test_ft.csv",
    index=False
)

cvd_train.to_csv(
    TRAIN_DIR / "cvd_train_ft.csv",
    index=False
)

cvd_test.to_csv(
    TEST_DIR / "cvd_test_ft.csv",
    index=False
)

print("T2D:", t2d_train.shape, t2d_test.shape)
print("CVD:", cvd_train.shape, cvd_test.shape)

T2D: (8926, 96) (2232, 96)
CVD: (8926, 111) (2232, 111)


In [39]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
BASE = Path(r"D:\2026\MedCare\ML\Dataset\Clean\Train")

T2D_FEATURES = [
    "AGEY_B", "HEIGHT", "WEIGHT", "PMBMI", "PMWAIST",
    "SMOKEV", "SMOKEN", "VGACTX", "MDACTX", "LTACTX",
    "DRINK", "DRINKD", "DRINKN",
    "SLEEP", "SLEEPE", "SLEEPFAL", "SLEEPSN", "SLEEPBR",
    "HIBP", "HIBPS", "HIBPE", "ARTHR", "ARTHRE", "ARTHRS",
    "DRUGS", "BPMED", "HIGHCHOL", "CHOLMED", "MENOPAUSE", "ITOT"
]

CVD_FEATURES = [
    "AGEY_B", "HEIGHT", "WEIGHT", "PMBMI", "PMWAIST",
    "BPSYS", "BPDIA", "BPPULS",
    "SMOKEV", "SMOKEN", "VGACTX", "MDACTX", "LTACTX",
    "DRINK", "DRINKD", "DRINKN",
    "SLEEP", "SLEEPE", "SLEEPFAL", "SLEEPSN", "SLEEPBR",
    "HIBP", "HIBPS", "HIBPE", "ARTHR", "ARTHRE", "ARTHRS",
    "DRUGS", "BPMED", "HIGHCHOL", "IS_SOB", "SOB",
    "CHOLMED", "MENOPAUSE", "ITOT"
]

COMMON_FE = [
    "FE_WtHR",
    "FE_BMI_Waist",
    "FE_Weekly_Drinks",
    "FE_Smoking_Status",
    "FE_Physical_Activity_Score",
    "FE_Sleep_Disturbance_Index",
    "FE_Log_Income"
]

CVD_FE = [
    "FE_Pulse_Pressure",
    "FE_MAP",
    "FE_RPP"
]


def engineer_wave(df, wave, disease):
    p = f"R{wave}"

    # Lấy đúng tên cột của wave
    c = lambda x: f"H{wave}ITOT" if x == "ITOT" else f"{p}{x}"

    # Thể trạng
    df[f"{p}_FE_WtHR"] = df[c("PMWAIST")] / (df[c("HEIGHT")] * 100)
    df[f"{p}_FE_BMI_Waist"] = df[c("PMBMI")] * df[c("PMWAIST")]

    # Rượu bia
    df[f"{p}_FE_Weekly_Drinks"] = (
        df[c("DRINKD")] * df[c("DRINKN")]
    ).where(df[c("DRINK")].eq(1), 0)

    # Hút thuốc
    df[f"{p}_FE_Smoking_Status"] = np.select(
        [
            df[c("SMOKEV")].eq(0),
            df[c("SMOKEV")].eq(1) & df[c("SMOKEN")].eq(0),
            df[c("SMOKEV")].eq(1) & df[c("SMOKEN")].eq(1)
        ],
        [0, 1, 2],
        default=np.nan
    )

    # Vận động
    df[f"{p}_FE_Physical_Activity_Score"] = (
        (6 - df[c("VGACTX")]) * 3
        + (6 - df[c("MDACTX")]) * 2
        + (6 - df[c("LTACTX")])
    )

    # Giấc ngủ
    df[f"{p}_FE_Sleep_Disturbance_Index"] = (
        df[c("SLEEP")]
        + df[c("SLEEPFAL")]
        + df[c("SLEEPSN")]
        + df[c("SLEEPBR")]
    )

    # Thu nhập
    df[f"{p}_FE_Log_Income"] = np.log1p(
        df[c("ITOT")].clip(lower=0)
    )


    # CVD-specific
    if disease == "CVD":
        df[f"{p}_FE_Pulse_Pressure"] = (
            df[c("BPSYS")] - df[c("BPDIA")]
        )

        df[f"{p}_FE_MAP"] = (
            df[c("BPDIA")]
            + df[f"{p}_FE_Pulse_Pressure"] / 3
        )

        df[f"{p}_FE_RPP"] = (
            df[c("BPSYS")] * df[c("BPPULS")]
        )

    return df


def process_file(filename, disease):
    df = pd.read_csv(BASE / filename)

    features = T2D_FEATURES if disease == "T2D" else CVD_FEATURES
    target = f"R16{disease}"

    # FE cho cả 3 wave
    for wave in [13, 14, 15]:
        df = engineer_wave(df, wave, disease)

    # Feature đầu vào
    cols = ["HHIDPN", "RAGENDER"]

    for wave in [13, 14, 15]:
        p = f"R{wave}"

        for feature in features:
            # ITOT dùng H13/H14/H15
            col = f"H{wave}ITOT" if feature == "ITOT" else f"{p}{feature}"
            cols.append(col)

        # FE
        cols += [f"{p}_{fe}" for fe in COMMON_FE]

        if disease == "CVD":
            cols += [f"{p}_{fe}" for fe in CVD_FE]

    # Target bệnh ở các wave quá khứ
    cols += [f"R{w}{disease}" for w in [13, 14, 15]]

    # Target tương lai R16
    cols.append(target)

    # Kiểm tra thiếu cột
    missing = [c for c in cols if c not in df.columns]

    if missing:
        raise ValueError(f"{disease} - Thiếu cột: {missing}")

    out = df[cols].copy()

    output = BASE / f"{disease.lower()}_train_ft_engineered.csv"
    out.to_csv(output, index=False)

    print(f"{disease}: {out.shape}")
    print(f"Saved: {output}")

    return out


t2d_future = process_file("t2d_train_ft.csv", "T2D")
cvd_future = process_file("cvd_train_ft.csv", "CVD")

T2D: (8926, 117)
Saved: D:\2026\MedCare\ML\Dataset\Clean\Train\t2d_train_ft_engineered.csv
CVD: (8926, 141)
Saved: D:\2026\MedCare\ML\Dataset\Clean\Train\cvd_train_ft_engineered.csv


In [40]:
BASE = Path(r"D:\2026\MedCare\ML\Dataset\Clean\Test")
def process_file(filename, disease):
    df = pd.read_csv(BASE / filename)

    features = T2D_FEATURES if disease == "T2D" else CVD_FEATURES
    target = f"R16{disease}"

    # FE cho cả 3 wave
    for wave in [13, 14, 15]:
        df = engineer_wave(df, wave, disease)

    # Feature đầu vào
    cols = ["HHIDPN", "RAGENDER"]

    for wave in [13, 14, 15]:
        p = f"R{wave}"

        for feature in features:
            # ITOT dùng H13/H14/H15
            col = f"H{wave}ITOT" if feature == "ITOT" else f"{p}{feature}"
            cols.append(col)

        # FE
        cols += [f"{p}_{fe}" for fe in COMMON_FE]

        if disease == "CVD":
            cols += [f"{p}_{fe}" for fe in CVD_FE]

    # Target bệnh ở các wave quá khứ
    cols += [f"R{w}{disease}" for w in [13, 14, 15]]

    # Target tương lai R16
    cols.append(target)

    # Kiểm tra thiếu cột
    missing = [c for c in cols if c not in df.columns]

    if missing:
        raise ValueError(f"{disease} - Thiếu cột: {missing}")

    out = df[cols].copy()

    output = BASE / f"{disease.lower()}_test_ft_engineered.csv"
    out.to_csv(output, index=False)

    print(f"{disease}: {out.shape}")
    print(f"Saved: {output}")

    return out


t2d_future = process_file("t2d_test_ft.csv", "T2D")
cvd_future = process_file("cvd_test_ft.csv", "CVD")

T2D: (2232, 117)
Saved: D:\2026\MedCare\ML\Dataset\Clean\Test\t2d_test_ft_engineered.csv
CVD: (2232, 141)
Saved: D:\2026\MedCare\ML\Dataset\Clean\Test\cvd_test_ft_engineered.csv
